# CPE rebuild on correct depth and head-camera intrinsics (Colab)

Stage 1 has to be re-run whenever its inputs change. Fixed so far (docs/architecture.md §9):
- **Depth** (2026-10-17): SANPO depth is a 2-value header + 720×1280 image, not 960×960.
- **Camera** (2026-10-19 run): fx comes from the head camera (ZED-M, ~1376 px), not the chest camera
  (~1974 px). fx sets every bearing and the detector's size check, and the size check now reads depth
  at the box centre scaled to the depth map. On one test walk this added 9% detections and widened
  bearings by 38%.

Everything is written to fresh Drive folders (`RUN_TAG` below), so earlier runs' CSVs are never reused;
`dataset/GRU/build.py` also refuses CSVs streamed with a non-head-camera fx.

| Part | What | Where it runs | Time (T4) |
|---|---|---|---|
| A | Stage 1 (YOLO26n v3 + ByteTrack + box depth) on all 462 walks | GPU | ~2–5 h |
| B1 | GRU arrival dataset → official split → corridor labels | CPU | ~10 min |
| B2 | Refit the Kalman noise on train (1 s-ahead likelihood), keep it if it beats the declared values on validation | CPU | ~30–60 min |
| B3 | Retrain the GRUs (4 variants × 3 seeds) | GPU | ~30–60 min |
| B4 | Baseline ladder, end-to-end replay (early-warning rules + announcer sweep), Kalman before/after | CPU | ~1–1.5 h |

**How to use**
1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*.
3. If Colab disconnects, reconnect and **Run all** again. Finished walks and finished steps
   are kept in Google Drive (`MyDrive/cpe_rebuild/`) and skipped, so it carries on where it stopped.
4. When the last cell prints `ALL DONE`, download `MyDrive/cpe_rebuild/cpe_rebuild_<RUN_TAG>_results.zip`
   and hand it back (unzip over the repo; then docs + CHANGELOG get updated and committed locally).

Nothing here pushes to GitHub. SANPO depth is used as is (no LiDAR simulation).

In [ ]:
# Settings
BRANCH = "v2-kalman-lidar"
REPO = "https://github.com/PES-112/CompositePerceptionEngine.git"
SEEDS = "0 1 2"
RUN_TAG = "headfx"                  # Drive folders and the results zip are named after this
REPLAY_NAME = "corridor_headfx"     # evaluation/benchmarks/threat_engine_replay/<this>

## Setup (safe to re-run)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, pathlib
DRIVE = pathlib.Path("/content/drive/MyDrive/cpe_rebuild")
CSV_DIR = DRIVE / f"sanpo_real_462_{RUN_TAG}"   # Stage-1 CSVs (one per walk) live on Drive
SAVED = DRIVE / f"outputs_{RUN_TAG}"              # Part B outputs, mirrored after every step
for d in (CSV_DIR, SAVED):
    d.mkdir(parents=True, exist_ok=True)
ROOT = pathlib.Path("/content/cpe")

In [ ]:
if not ROOT.exists():
    !git clone --branch {BRANCH} --single-branch --depth 1 {REPO} {ROOT}
else:
    !git -C {ROOT} pull --ff-only
%cd {ROOT}
!git log --oneline -1

# Colab already has torch (CUDA), numpy, pandas, scipy, opencv, pillow.
os.environ["YOLO_AUTOINSTALL"] = "false"   # stop ultralytics from re-pinning numpy behind our back
!pip -q install "ultralytics==8.4.38" "lap>=0.5.12"

# The build names its dataset after the CSV folder, so keep the original name: sanpo_real_462.
!mkdir -p data/processed
!ln -sfn {CSV_DIR} data/processed/sanpo_real_462

# Restore Part B outputs from an earlier Colab session, if any.
!rsync -a {SAVED}/ {ROOT}/

In [ ]:
import torch, ultralytics, numpy
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| ultralytics", ultralytics.__version__, "| numpy", numpy.__version__)
# The GPU is needed for Stage 1 and GRU training only; once both are done, a CPU runtime finishes the rest.
gpu_work_left = len(list(CSV_DIR.glob("*.csv"))) < 440 or not (SAVED / ".done" / "b3_gru").exists()
if gpu_work_left:
    assert torch.cuda.is_available(), "No GPU - Stage 1 would take a day on CPU. Switch the runtime to T4 GPU."
elif not torch.cuda.is_available():
    print("No GPU, but Stage 1 and the GRUs are done: the remaining steps run on CPU.")

## Smoke test (1 walk, 20 frames, ~1 min)

Checks the GPU path and that depth is decoded right: distances should be a few metres, not 18 m to a person next to you.

In [ ]:
!rm -rf /content/smoke
!python tools/stream_sanpo_perception.py --session-fraction 1.0 --max-sessions 1 --max-frames 20 \
    --out-dir /content/smoke --scratch-dir /content/scratch_smoke
import pandas as pd, glob
df = pd.concat(pd.read_csv(f) for f in glob.glob("/content/smoke/*.csv"))
print(df.groupby("class")["distance_m"].describe()[["count", "50%", "min", "max"]])

## Part A: Stage 1 on all 462 walks (the long one)

Each walk is downloaded to local scratch, run through YOLO + tracking + depth, written to Drive as
`<session>.csv`, then deleted. Walks that already have a CSV are skipped, so re-running resumes.
The full log is in `MyDrive/cpe_rebuild/stage1.log`.

In [ ]:
!python tools/stream_sanpo_perception.py --session-fraction 1.0 --max-frames 300 --frame-stride 3 \
    --download-workers 16 --out-dir data/processed/sanpo_real_462 --scratch-dir /content/scratch \
    2>&1 | tee -a {DRIVE}/stage1.log | grep --line-buffered -E "^\[|FAILED|Done|Sampled"

In [ ]:
n = len(list(CSV_DIR.glob("*.csv")))
print(f"{n}/462 walks have a CSV")
if n < 462:
    print("Some walks failed (network blips). Re-run the Part A cell: it only retries the missing ones.")
    !grep -B1 FAILED {DRIVE}/stage1.log | tail -20

## Part B: rebuild datasets, refit, retrain, re-score

Each step writes a marker in `MyDrive/cpe_rebuild/outputs/.done/` and is skipped next time.

In [ ]:
import shlex, shutil, time

OUTPUT_PATHS = [
    "dataset/GRU/sanpo_real_462_tracked_d3_h8_current_m32_f5_s0",
    "dataset/GRU/sanpo_real_462_tracked_d3_h8_current_m32_official",
    *[f"dataset/GRU/sanpo_real_462_tracked_d3_h8_current_m32_official_corridor_w{w}_d3_h{h}"
      for w in ("0.75", "1.5") for h in (13, 20, 25)],
    "src/threat_engine/fusion.py",
    "evaluation/benchmarks/kalman_noise_fit",
    "models/GRU/GRU/runs", "models/GRU/GRU_attention/runs",
    "evaluation/benchmarks/baseline_ladder",
    "evaluation/benchmarks/threat_engine_replay",
]
DONE = SAVED / ".done"
DONE.mkdir(parents=True, exist_ok=True)


def save():
    for p in OUTPUT_PATHS:
        if (ROOT / p).exists():
            dest = SAVED / p
            dest.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", str(ROOT / p) + ("/" if (ROOT / p).is_dir() else ""), str(dest) + ("/" if (ROOT / p).is_dir() else "")], check=True)


def step(name, *commands):
    if (DONE / name).exists():
        print(f"[skip] {name} (done in an earlier session)")
        return
    t = time.time()
    for c in commands:
        print(f"$ {c}", flush=True)
        subprocess.run(c, shell=True, check=True, cwd=ROOT)
    save()
    (DONE / name).write_text(time.strftime("%Y-%m-%d %H:%M"))
    print(f"[done] {name} in {(time.time() - t) / 60:.1f} min")


DS = "dataset/GRU/sanpo_real_462_tracked_d3_h8_current_m32"
CORRIDOR = f"{DS}_official_corridor_w0.75_d3_h20"

### B1: arrival dataset, official split, corridor labels

In [ ]:
step("b1_datasets",
     "python -m dataset.GRU.build --csv-dir data/processed/sanpo_real_462 --tracked-only --hazard-m 3 --folds 5 --overwrite",
     f"python -m dataset.GRU.resplit --dataset {DS}_f5_s0 --overwrite",
     *[f"python -m dataset.GRU.relabel --dataset {DS}_official --width {w} --depth 3 --horizon {h} --overwrite"
       for w in ("0.75", "1.5") for h in (13, 20, 25)])

### B2: refit the Kalman noise on train (1 s-ahead likelihood); keep it only if it beats the declared values on validation

In [ ]:
import json, re

FIELDS = ("accel_std", "static_accel_std", "depth_base", "depth_lin", "depth_quad", "bearing_std_deg",
          "init_speed_std", "robust_nu", "gate_nis")

def apply_fit():
    out = json.loads((ROOT / "evaluation/benchmarks/kalman_noise_fit/fit_pose_ahead.json").read_text())
    val = {name: out["scores"][name]["val"]["ahead_ll"] for name in ("declared", "fitted")}
    chosen = max(val, key=val.get)
    print("validation 1 s-ahead log-likelihood:", val, "-> using", chosen)
    noise = out[chosen]
    src = (ROOT / "src/threat_engine/fusion.py").read_text()
    for k in FIELDS:
        src, n = re.subn(rf"^(    {k}: float = )[0-9.eE+-]+$", lambda m: m.group(1) + repr(float(f"{noise[k]:.3g}")), src, flags=re.M)
        assert n == 1, k
    (ROOT / "src/threat_engine/fusion.py").write_text(src)
    print("fusion.py NoiseModel defaults <-", {k: round(noise[k], 3) for k in FIELDS})

if not (DONE / "b2_kalman").exists():
    subprocess.run("python tools/fit_kalman_noise.py --objective ahead", shell=True, check=True, cwd=ROOT)
    apply_fit()
step("b2_kalman", "python -m pytest -q tests/test_kalman_latency.py tests/test_threat_engine.py")
!grep -n -A13 "Fitted 20" src/threat_engine/fusion.py

### B3: retrain the GRUs (same settings as the 2026-10-17 runs)

In [ ]:
step("b3_gru",
     f"python -m models.GRU.train --arch gru --dataset {CORRIDOR} --time-bins 4 --seeds {SEEDS} --run-name corridor_h8 --overwrite",
     f"python -m models.GRU.train --arch gru_attention --dataset {CORRIDOR} --time-bins 4 --seeds {SEEDS} --run-name corridor_h8 --overwrite",
     f"python -m models.GRU.train --arch gru --dataset {CORRIDOR} --time-bins 4 --history-steps 4 --seeds {SEEDS} --run-name corridor_hist4 --overwrite",
     f"python -m models.GRU.train --arch gru --dataset {CORRIDOR} --time-bins 4 --history-steps 1 --seeds {SEEDS} --run-name corridor_hist1 --overwrite")

### B4: baseline ladder, end-to-end replay, Kalman before/after (test scored once)

In [ ]:
RUNS = "--run gru=corridor_h8:gru --run gru_attention=corridor_h8:gru_attention --run gru_hist4=corridor_hist4:gru --run gru_hist1=corridor_hist1:gru"
LADDER = f"evaluation/benchmarks/baseline_ladder/{pathlib.Path(CORRIDOR).name}"
step("b4_ladder",
     f"python -m models.GRU.benchmark --dataset {CORRIDOR} {RUNS}",
     f"python -m models.GRU.benchmark --dataset {CORRIDOR} {RUNS} --reference cv_kalman_declared --out-dir {LADDER}/ref_cv_kalman_declared")
step("b4_replay",
     f"python tools/replay_threat_engine.py --dataset {CORRIDOR} --gru-run models/GRU/GRU/runs/corridor_h8_seed0 --name {REPLAY_NAME} "
     "--announcer-sweep v2_gru v2_kalman nearest_in_corridor")
step("b4_kalman_before_after",
     "python tools/compare_kalman_noise.py --split test")

## Package the results

In [ ]:
# Needs the setup cells above in this session (use Run all; finished work is skipped).
import pathlib, subprocess
DRIVE = pathlib.Path("/content/drive/MyDrive/cpe_rebuild")
SAVED = DRIVE / f"outputs_{RUN_TAG}"
n_csv = len(list((DRIVE / f"sanpo_real_462_{RUN_TAG}").glob("*.csv")))
steps = ["b1_datasets", "b2_kalman", "b3_gru", "b4_ladder", "b4_replay", "b4_kalman_before_after"]
missing = [s for s in steps if not (SAVED / ".done" / s).exists()]
print(f"Stage 1: {n_csv}/462 walks")
print("Part B steps done:", [s for s in steps if s not in missing], "| missing:", missing)
if missing:
    print("NOT DONE: run Runtime -> Run all again (it resumes), then check this cell.")
else:
    zip_path = DRIVE / f"cpe_rebuild_{RUN_TAG}_results.zip"
    zip_path.unlink(missing_ok=True)
    subprocess.run(["zip", "-qr", str(zip_path), ".", "-x", ".done/*"], cwd=SAVED, check=True)
    print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    for rep in SAVED.glob("evaluation/benchmarks/baseline_ladder/*corridor_w0.75_d3_h20/report.md"):
        print("".join(rep.read_text().splitlines(True)[:30]))
    print(f"ALL DONE: download MyDrive/cpe_rebuild/{zip_path.name}")